# Reasoning Models 推理模型实验室

本 notebook 模拟 reasoning model 的核心机制：
1. **GRPO (Group Relative Policy Optimization)** 算法模拟
2. **Test-time Compute Scaling** 实验
3. **Best-of-N vs Chain-of-Thought** 对比
4. **MLA KV Cache 压缩**模拟

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Tuple, Optional
import matplotlib
matplotlib.rcParams['font.size'] = 11
np.random.seed(42)
print('Environment ready.')

## 1. GRPO (Group Relative Policy Optimization) 模拟

GRPO 是 DeepSeek-R1 使用的 RL 算法，核心是**组内相对排序**：
- 对同一 prompt 采样 G 个回答
- 计算每个回答的 reward
- Advantage = (r_i - mean) / std（组内归一化）
- 不需要 critic network（比 PPO 省一半显存）

In [ ]:
class SimplePolicy:
    def __init__(self, dim=10):
        self.weights = np.random.randn(dim) * 0.1
        self.dim = dim

    def log_prob(self, prompt_feat, response_quality):
        mean = np.dot(self.weights, prompt_feat)
        return -0.5 * (response_quality - mean) ** 2

    def sample(self, prompt_feat, n=1):
        mean = np.dot(self.weights, prompt_feat)
        return np.random.normal(mean, 1.0, size=n)

    def update(self, gradient, lr):
        self.weights += lr * gradient

def grpo_step(policy, old_policy, prompt_feat, reward_fn,
              group_size=8, clip_eps=0.2, kl_beta=0.01, lr=0.01):
    G = group_size
    responses = old_policy.sample(prompt_feat, n=G)
    rewards = np.array([reward_fn(r) for r in responses])
    advantage = (rewards - rewards.mean()) / (rewards.std() + 1e-8)
    grad = np.zeros_like(policy.weights)
    for i in range(G):
        new_lp = policy.log_prob(prompt_feat, responses[i])
        old_lp = old_policy.log_prob(prompt_feat, responses[i])
        ratio = np.exp(new_lp - old_lp)
        clipped = np.clip(ratio, 1 - clip_eps, 1 + clip_eps)
        surr = min(ratio * advantage[i], clipped * advantage[i])
        diff = responses[i] - np.dot(policy.weights, prompt_feat)
        grad += surr * diff * prompt_feat
    grad /= G
    grad -= kl_beta * (policy.weights - old_policy.weights)
    policy.update(grad, lr)
    return rewards.mean(), rewards.max()

print('GRPO components defined.')

In [ ]:
def math_reward(response_quality):
    return 1.0 if response_quality > 1.5 else 0.0

policy = SimplePolicy(dim=5)
prompt = np.random.randn(5)
history = {'mean_r': [], 'max_r': [], 'quality': []}

for step in range(200):
    old_policy = SimplePolicy(dim=5)
    old_policy.weights = policy.weights.copy()
    mean_r, max_r = grpo_step(policy, old_policy, prompt, math_reward,
                              group_size=16, lr=0.005)
    qualities = policy.sample(prompt, n=50)
    history['mean_r'].append(mean_r)
    history['max_r'].append(max_r)
    history['quality'].append(qualities.mean())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['mean_r'], label='Mean Reward', alpha=0.7)
ax1.plot(history['max_r'], label='Max Reward', alpha=0.7)
ax1.set_xlabel('Training Step'); ax1.set_ylabel('Reward')
ax1.set_title('GRPO Training: Reward Curve'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history['quality'], color='green')
ax2.axhline(y=1.5, color='red', linestyle='--', label='Reward Threshold')
ax2.set_xlabel('Training Step'); ax2.set_ylabel('Mean Response Quality')
ax2.set_title('GRPO: Policy Quality Improvement'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print('GRPO training complete. Policy learned to exceed reward threshold.')

## 2. Test-Time Compute Scaling

核心发现：增加推理时计算量 -> 性能提升
- **Sequential**：更长的思考链
- **Parallel**：Best-of-N 采样

In [ ]:
def simulate_problem_solving(difficulty, thinking_budget, n_samples=1):
    base_ability = 5.0
    results = []
    for _ in range(n_samples):
        insight = sum(np.random.exponential(0.3) for _ in range(thinking_budget))
        effective_ability = base_ability + np.log1p(insight)
        solve_prob = 1.0 / (1.0 + np.exp(-(effective_ability - difficulty)))
        results.append(np.random.random() < solve_prob)
    return any(results)

difficulties = [3, 5, 7, 9]
thinking_budgets = [1, 2, 5, 10, 20, 50]
n_samples_list = [1, 2, 5, 10, 20, 50]
n_trials = 500

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
for diff in difficulties:
    accs = []
    for budget in thinking_budgets:
        success = sum(simulate_problem_solving(diff, budget, 1) for _ in range(n_trials))
        accs.append(success / n_trials)
    ax.plot(thinking_budgets, accs, 'o-', label=f'Difficulty={diff}')
ax.set_xlabel('Thinking Budget (sequential)'); ax.set_ylabel('Solve Rate')
ax.set_title('Sequential Scaling: Longer CoT'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
for diff in difficulties:
    accs = []
    for ns in n_samples_list:
        success = sum(simulate_problem_solving(diff, 5, ns) for _ in range(n_trials))
        accs.append(success / n_trials)
    ax.plot(n_samples_list, accs, 's-', label=f'Difficulty={diff}')
ax.set_xlabel('Num Samples (parallel)'); ax.set_ylabel('Solve Rate')
ax.set_title('Parallel Scaling: Best-of-N'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print('Both sequential and parallel scaling improve solve rate.')

## 3. Reward Design: Rule vs Model

- **Rule Reward** (R1): 答案正确 -> 1，错误 -> 0
- **Process Reward Model (PRM)** (o1): 每步打分

In [ ]:
np.random.seed(42)
n_episodes = 300
window = 20

rule_rewards, prm_rewards = [], []
for ep in range(n_episodes):
    skill = min(ep / 200, 1.0)
    n_steps = np.random.randint(3, 8)
    final_correct = np.random.random() < (0.3 + 0.6 * skill)
    rr = 1.0 if final_correct else 0.0
    rule_rewards.append(rr)
    step_quals = [max(0, np.random.normal(skill * 0.7, 0.3)) for _ in range(n_steps)]
    pr = (sum(step_quals) + (2.0 if final_correct else 0.0)) / (n_steps + 2)
    prm_rewards.append(pr)

def smooth(arr, w):
    return [np.mean(arr[max(0,i-w):i+1]) for i in range(len(arr))]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(smooth(rule_rewards, window), label='Rule Reward (R1-style)', linewidth=2)
ax.plot(smooth(prm_rewards, window), label='Process Reward (o1-style)', linewidth=2)
ax.set_xlabel('Training Episode'); ax.set_ylabel('Smoothed Reward')
ax.set_title('Rule Reward vs Process Reward Model (PRM)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print('PRM converges faster but Rule Reward needs ZERO annotation cost.')
print('DeepSeek-R1 proved Rule Reward + GRPO is sufficient!')

## 4. MLA (Multi-head Latent Attention) KV Cache 压缩模拟

DeepSeek-V3 的 MLA：将 KV 投影到低维 latent space，只缓存 latent vector。

In [ ]:
def simulate_mla_compression(seq_len, d_model, n_heads, d_latent, n_layers):
    d_head = d_model // n_heads
    mha_kv = 2 * n_heads * d_head * seq_len * n_layers * 2
    n_groups = 8
    gqa_kv = 2 * n_groups * d_head * seq_len * n_layers * 2
    mla_kv = 2 * d_latent * seq_len * n_layers * 2
    return {'MHA': mha_kv/1e9, 'GQA-8': gqa_kv/1e9, f'MLA(d={d_latent})': mla_kv/1e9}

configs = [
    {'seq_len': 4096, 'label': '4K'},
    {'seq_len': 32768, 'label': '32K'},
    {'seq_len': 131072, 'label': '128K'},
]
d_model, n_heads, d_latent, n_layers = 7168, 128, 512, 61

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(configs))
width = 0.25
for i, method_name in enumerate(['MHA', 'GQA-8', f'MLA(d={d_latent})']):
    vals = []
    for cfg in configs:
        result = simulate_mla_compression(cfg['seq_len'], d_model, n_heads, d_latent, n_layers)
        vals.append(list(result.values())[i])
    bars = ax.bar(x + i * width, vals, width, label=method_name)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
               f'{v:.1f}GB', ha='center', fontsize=8)

ax.set_xlabel('Sequence Length'); ax.set_ylabel('KV Cache Size (GB)')
ax.set_title('KV Cache: MHA vs GQA vs MLA (DeepSeek-V3 config)')
ax.set_xticks(x + width)
ax.set_xticklabels([c['label'] for c in configs])
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

result = simulate_mla_compression(131072, d_model, n_heads, d_latent, n_layers)
mha_v = list(result.values())[0]
mla_v = list(result.values())[2]
print(f'At 128K tokens:')
print(f'  MHA KV Cache: {mha_v:.1f} GB')
print(f'  MLA KV Cache: {mla_v:.1f} GB')
print(f'  Compression ratio: {mha_v/mla_v:.1f}x')

## Summary

| 技术 | 核心思想 | 代表模型 |
|------|---------|----------|
| GRPO | 组内相对排序，无需 critic | DeepSeek-R1 |
| Test-time Compute | 推理时增加计算提升质量 | o1, R1, k1.5 |
| Rule Reward | 零标注成本的 RL 训练 | DeepSeek-R1 |
| PRM | 过程监督，收敛更快 | OpenAI o1 |
| MLA | KV Cache 低秩压缩 | DeepSeek-V3 |

### 面试关键句
- GRPO 的核心是用组内相对排序替代 critic network，省一半显存
- R1 证明了纯 RL + 规则 reward 就能涌现推理能力
- MLA 本质是在 attention 层做 learned projection，压缩 KV 表示
- Test-time compute scaling 是 2025 年最重要的范式转变